In [2]:
import plotly.graph_objects as go
import numpy as np
from typing import Dict, List, Tuple, Optional
import json
from pathlib import Path
import plotly.express as px


def load_evaluation_data(
    eval_type: str,
    base_dir: str,
    model_id: str
) -> List[Tuple[float, float]]:
    """Load evaluation data from JSON files."""
    base_name = model_id.split("/")[-1]
    model_dir = Path(base_dir) / base_name
    
    if not model_dir.exists():
        raise FileNotFoundError(f"Directory not found: {model_dir}")
    
    points = []
    for json_file in sorted(model_dir.glob("*.json")):
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
            degree = data.get("degree")
            perplexity = data.get("metadata", {}).get("average_perplexity")
            if degree is not None and perplexity is not None:
                points.append((degree, perplexity))
        except (json.JSONDecodeError, KeyError):
            continue
    return sorted(points, key=lambda x: x[0])


def load_all_evaluation_data(
    eval_configs: Dict[str, str],
    model_id: str
) -> Dict[str, List[Tuple[float, float]]]:
    """Load data from multiple evaluation types."""
    data_dict = {}
    for eval_type, base_dir in eval_configs.items():
        try:
            data_dict[eval_type] = load_evaluation_data(eval_type, base_dir, model_id)
        except FileNotFoundError:
            data_dict[eval_type] = []
    return data_dict


def get_baseline_perplexity(data_dict: Dict[str, List[Tuple[float, float]]]) -> Optional[float]:
    """Get baseline from Selective at degree 0."""
    if "Selective" not in data_dict:
        return None
    for degree, perplexity in data_dict["Selective"]:
        if degree == 0:
            return perplexity
    return None


def plot_spider_charts_paper(
    all_data: List[Dict[str, List[Tuple[float, float]]]],
    model_names: List[str],
    max_perplexity: float = 25.0,
    save_filename: str = "spider_charts.png",
    title: str = None,
    n_cols: int = 2,
    fig_width: int = 800,
    fig_height: int = 1000,
    scale: float = 3.0,
):
    """
    Plot spider charts in paper format.
    
    Args:
        all_data: List of data dicts for each model
        model_names: List of model names
        max_perplexity: Maximum value for radial axis
        save_filename: Output filename
        title: Optional main title
        n_cols: Number of columns in grid
        fig_width: Figure width in pixels
        fig_height: Figure height in pixels
        scale: Scale factor for output resolution
    """
    n_models = len(all_data)
    n_rows = (n_models + n_cols - 1) // n_cols
    
    # # Colors matching reference style
    # colors = {
    #     "Standard": "#1E90FF",    # Blue
    #     "Adaptive": "#00CED1",    # Cyan
    #     "Selective": "#DC143C",   # Red
    # }
    # Generate automatic colors using Plotly's color palette
    n_series = len(all_data[0])
    colors = px.colors.qualitative.Set2[:n_series] if n_series <= len(px.colors.qualitative.Set2) else px.colors.qualitative.Plotly
    colors = {k: v for k, v in zip(all_data[0].keys(), colors)}
    
    # Line properties
    line_width = 2.5
    baseline_width = 2
    
    fig = go.Figure()
    
    # Calculate domains for grid - tight spacing
    margin_x = 0.02
    margin_y = 0.08
    gap_x = 0.04
    gap_y = 0.06
    
    plot_w = (1.0 - 2 * margin_x - (n_cols - 1) * gap_x) / n_cols
    plot_h = (1.0 - margin_y - (n_rows - 1) * gap_y - 0.04) / n_rows
    
    domains = []
    for row in range(n_rows):
        for col in range(n_cols):
            x0 = margin_x + col * (plot_w + gap_x)
            x1 = x0 + plot_w
            y1 = 1.0 - 0.02 - row * (plot_h + gap_y)
            y0 = y1 - plot_h
            domains.append({"x": [x0, x1], "y": [y0, y1]})
    
    legend_added = {k: False for k in colors.keys()}
    legend_added["Baseline"] = False
    
    # Angular ticks - every 10 degrees
    angular_ticks = np.arange(0, 360, 10)
    
    # Calculate radial range based on max_perplexity
    radial_max = max_perplexity * 1.2
    radial_ticks = np.linspace(0, max_perplexity, 6)
    
    for idx, (data_dict, model_name) in enumerate(zip(all_data, model_names)):
        if idx >= len(domains):
            break
            
        polar_key = f"polar{idx + 1}" if idx > 0 else "polar"
        
        # Get all degrees from data
        all_degrees = set()
        for points in data_dict.values():
            for degree, _ in points:
                all_degrees.add(degree)
        degrees = np.array(sorted(all_degrees))
        
        # Get baseline
        baseline = get_baseline_perplexity(data_dict)
        
        # Add background fill circle first (so it's behind everything)
        theta_bg = np.arange(0, 360 + 1, 10)
        fig.add_trace(go.Scatterpolar(
            r=np.full_like(theta_bg, radial_max),
            theta=theta_bg,
            mode='lines',
            fill='toself',
            fillcolor='rgba(240, 248, 255, 0.5)',  # Light blue background
            line=dict(color='rgba(0,0,0,0)', width=0),
            showlegend=False,
            subplot=polar_key,
        ))
        
        # Plot each method (no fill, just lines)
        for eval_type, points in data_dict.items():
            if not points or eval_type not in colors:
                continue
            
            point_dict = {d: p for d, p in points}
            perplexities = np.array([point_dict.get(d, np.nan) for d in degrees])
            
            sort_idx = np.argsort(degrees)
            theta_sorted = degrees[sort_idx]
            perp_sorted = perplexities[sort_idx]
            
            # Cap values at max_perplexity for plotting
            perp_capped = np.minimum(perp_sorted, max_perplexity)
            
            # Close circle
            theta_closed = np.append(theta_sorted, theta_sorted[0])
            perp_closed = np.append(perp_capped, perp_capped[0])
            
            show_legend = not legend_added[eval_type]
            legend_added[eval_type] = True
            
            fig.add_trace(go.Scatterpolar(
                r=perp_closed,
                theta=theta_closed,
                mode='lines',
                name=eval_type,
                line=dict(color=colors[eval_type], width=line_width),
                showlegend=show_legend,
                legendgroup=eval_type,
                subplot=polar_key,
            ))
            
            # Add red stars for values exceeding max_perplexity
            exceed_mask = perp_sorted > max_perplexity
            if np.any(exceed_mask):
                exceed_theta = theta_sorted[exceed_mask]
                fig.add_trace(go.Scatterpolar(
                    r=np.full_like(exceed_theta, max_perplexity, dtype=float),
                    theta=exceed_theta,
                    mode='markers',
                    marker=dict(
                        symbol='star',
                        size=15,
                        color='red',
                        line=dict(color='black', width=1)
                    ),
                    showlegend=False,
                    subplot=polar_key,
                ))
        
        # Baseline circle
        if baseline is not None:
            theta_circle = np.arange(0, 360 + 1, 10)
            show_legend = not legend_added["Baseline"]
            legend_added["Baseline"] = True
            
            fig.add_trace(go.Scatterpolar(
                r=np.full_like(theta_circle, baseline, dtype=float),
                theta=theta_circle,
                mode='lines',
                name='Baseline',
                # line=dict(color='gray', width=baseline_width, dash='dot'),
                line=dict(color='gray', width=3, dash='dot'),
                opacity=0.3,    
                showlegend=show_legend,
                legendgroup="Baseline",
                subplot=polar_key,
            ))
        
        fig.update_layout(**{
            polar_key: dict(
                domain=domains[idx],
                radialaxis=dict(
                    visible=True,
                    range=[0, radial_max],
                    tickvals=radial_ticks,
                    ticktext=[f'{t:.1f}' for t in radial_ticks],
                    tickfont=dict(size=14, color='#666'),
                    gridcolor='#d0d0d0',
                    gridwidth=0.5,
                    linewidth=0.5,
                    linecolor='#d0d0d0',
                ),
                angularaxis=dict(
                    tickvals=angular_ticks,
                    ticktext=[f'{int(d)}°' for d in angular_ticks],
                    tickfont=dict(size=16, color='#666'),
                    direction='counterclockwise',
                    rotation=0,
                    gridcolor='#d0d0d0',
                    gridwidth=0.3,
                    linewidth=0.5,
                    linecolor='#d0d0d0',
                ),
                bgcolor='white',
            )
        })
    
    # Add subplot titles
    short_names = [n.split("/")[-1] for n in model_names]
    annotations = []
    for idx, name in enumerate(short_names):
        if idx >= len(domains):
            break
        domain = domains[idx]
        annotations.append(dict(
            text=f"<b>{name}</b>",
            x=(domain["x"][0] + domain["x"][1]) / 2,
            y=domain["y"][1] + 0.02,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=18, color='black', family="Arial"),
            xanchor="center",
            yanchor="bottom",
        ))
    
    # Main title (optional)
    if title:
        annotations.append(dict(
            text=f"<b>{title}</b>",
            x=0.5,
            y=1.01,
            xref="paper",
            yref="paper",
            showarrow=False,
            font=dict(size=24, color='black', family="Arial"),
            xanchor="center",
            yanchor="bottom",
        ))
    
    fig.update_layout(
        annotations=annotations,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="top",
            y=0.05,
            xanchor="center",
            x=0.5,
            font=dict(size=18),
            bgcolor='rgba(255,255,255,0.95)',
            borderwidth=0,
            itemsizing='constant',
            tracegroupgap=5,
        ),
        width=fig_width,
        height=fig_height,
        paper_bgcolor='white',
        plot_bgcolor='white',
        font=dict(family="Arial", size=10),
        margin=dict(t=30, b=60, l=20, r=20),
    )
    
    # Save
    try:
        fig.write_image(save_filename, width=fig_width, height=fig_height, scale=scale)
        print(f"✅ Figure saved as {save_filename}")
    except Exception as e:
        print(f"⚠️ Error: {e}")
    
    return fig


# Configuration
EVAL_CONFIGS = {
    "Standard": "../logs/perplexity/standard",
    "Adaptive": "../logs/perplexity/adaptive",
    "Selective": "../logs/perplexity/selective",
}

MODEL_IDS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
    "meta-llama/Llama-3.2-3B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-2-2b-it",
    "google/gemma-2-9b-it"
]


# Use demo data for testing
all_data = [load_all_evaluation_data(EVAL_CONFIGS, MODEL_ID) for MODEL_ID in MODEL_IDS]
model_names = MODEL_IDS

fig = plot_spider_charts_paper(
    all_data=all_data,
    model_names=model_names,
    max_perplexity=2.0,  # Actual perplexity scale
    save_filename="spider_charts.png",
    title=None,
    n_cols=4,
    fig_width=2000,    # Customize width
    fig_height=1400,  # Customize height
    scale=3.0,  
)

print("✅ Done!")

✅ Figure saved as spider_charts.png
✅ Done!
